# Authentication：在 LangGraph 平台上实现自定义认证、资源授权和连接认证提供者聊天机器人 示例

在 LangGraph 中实现自定义认证、资源授权和连接认证提供者
本系列教程将指导你构建一个具有安全认证和资源授权的聊天机器人，确保只有特定用户可以访问，并为每个用户提供私密的对话环境。我们将从 LangGraph 模板开始，逐步添加基于令牌的认证、资源级访问控制，并最终通过 OAuth2 集成 Supabase 等认证提供者，实现生产级的安全用户账户管理。

本系列分为三个部分：

- 设置自定义认证：控制谁可以访问你的机器人。
- 使对话私有：让用户拥有私密的对话。
- 连接认证提供者：添加真实用户账户并使用 OAuth2 进行生产级验证。


本教程假设你对以下概念有基本了解：

- LangGraph 平台的基本使用
- Python 异步编程（async/await）
- HTTP 请求和认证基础（例如 Bearer 令牌）
- OAuth2 认证流程

注意：自定义认证仅适用于 LangGraph 平台 SaaS 部署或企业自托管部署。

## 第 1 部分：设置自定义认证
1.1 创建你的应用

首先，使用 LangGraph 启动模板创建一个新的聊天机器人项目：
```json
pip install -U "langgraph-cli[inmem]"
langgraph new --template=new-langgraph-project-python custom-auth
cd custom-auth
```
该模板提供了一个占位符 LangGraph 应用。安装本地依赖并运行开发服务器以试运行：

```shell
pip install -e .
langgraph dev

```

服务器将启动并在浏览器中打开 LangGraph Studio：
```shell
> - 🚀 API: http://127.0.0.1:2024
> - 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
> - 📚 API Docs: http://127.0.0.1:2024/docs
> 
> 此内存服务器专为开发和测试设计。
> 对于生产使用，请使用 LangGraph 平台。

```

如果将此服务器托管在公共互联网上，任何人都可以访问它，这是不安全的！

<img src="https://i-blog.csdnimg.cn/direct/c5383ed3c587417d8b59d94d3db4d8e3.png">

### 1.2 添加认证
现在为你的 LangGraph 应用添加认证机制。

> 注意：在本教程中，你将使用硬编码的令牌作为示例。在第三部分中，你将实现生产级的认证方案。

`Auth` 对象允许你注册一个认证函数，LangGraph 平台将在每个请求上运行该函数。此函数接收请求并决定是否接受或拒绝。

创建一个新文件 `src/security/auth.py`，用于检查用户是否被允许访问你的机器人：

`src/security/auth.py`





In [1]:
from langgraph_sdk import Auth

# 这是我们的测试用户数据库。生产环境中不要这样做
VALID_TOKENS = {
    "user1-token": {"id": "user1", "name": "Alice"},
    "user2-token": {"id": "user2", "name": "Bob"},
}

# `Auth` 对象是 LangGraph 用于标记认证函数的容器
auth = Auth()

# `authenticate` 装饰器告诉 LangGraph 将此函数作为中间件在每个请求上调用
# 以确定请求是否被允许
@auth.authenticate
async def get_current_user(authorization: str | None) -> Auth.types.MinimalUserDict:
    """检查用户的令牌是否有效。"""
    assert authorization
    scheme, token = authorization.split()
    assert scheme.lower() == "bearer"
    # 检查令牌是否有效
    if token not in VALID_TOKENS:
        raise Auth.exceptions.HTTPException(status_code=401, detail="无效令牌")

    # 如果有效，返回用户信息
    user_data = VALID_TOKENS[token]
    return {
        "identity": user_data["id"],
    }



注意，你的认证处理程序执行了两个重要操作：

1. 检查请求的 Authorization 头中是否提供了有效令牌。
2. 返回用户的身份信息。

现在通过在 langgraph.json 配置中添加以下内容，告诉 LangGraph 使用认证：

langgraph.json

```json
{
  "dependencies": ["."],
  "graphs": {
    "agent": "./src/agent/graph.py:graph"
  },
  "env": ".env",
  "auth": {
    "path": "src/security/auth.py:auth"
  }
}
```

1.3 测试你的机器人
再次启动服务器以测试所有功能：

`langgraph dev --no-browser`

如果你未添加 `--no-browser`，Studio UI 将在浏览器中打开。你可能会疑惑，为什么 Studio 仍能连接到我们的服务器？默认情况下，即使使用自定义认证，我们也允许来自 LangGraph Studio 的访问。这便于在 Studio 中开发和测试你的机器人。你可以通过在认证配置中设置 `disable_studio_auth: "true"` 来移除此替代认证选项：

```json
{
    "auth": {
        "path": "src/security/auth.py:auth",
        "disable_studio_auth": "true"
    }
}

```



#### 1.4 与你的机器人聊天
现在，你只有在请求头中提供有效令牌时才能访问机器人。然而，在添加资源授权处理程序之前，用户仍能访问彼此的资源（将在下一部分解决）。

<img src="https://i-blog.csdnimg.cn/direct/e0e26d4ce26f4143a2abd08b9eb655d9.png">

在一个文件或notebook中运行以下代码：


In [ ]:
from langgraph_sdk import get_client

# 尝试不带令牌访问（应失败）
client = get_client(url="http://localhost:2024")
try:
    thread = await client.threads.create()
    print("❌ 没有令牌时应该失败！")
except Exception as e:
    print("✅ 正确阻止了访问：", e)

# 使用有效令牌尝试
client = get_client(
    url="http://localhost:2024", headers={"Authorization": "Bearer user1-token"}
)

# 创建线程并聊天
thread = await client.threads.create()
print(f"✅ 以 Alice 身份创建了线程：{thread['thread_id']}")

response = await client.runs.create(
    thread_id=thread["thread_id"],
    assistant_id="agent",
    input={"messages": [{"role": "user", "content": "Hello!"}]},
)
print("✅ 机器人响应：")
print(response)


你应该看到：

1. 没有有效令牌时，无法访问机器人。
2. 使用有效令牌时，可以创建线程并进行聊天。

总结：恭喜！你已经构建了一个只允许“认证”用户访问的聊天机器人。虽然此系统尚未实现生产级的安全方案，但我们已经学习了控制机器人访问的基本机制。在下一部分，我们将学习如何为每个用户提供私密的对话。

> 现在你可以控制谁访问你的机器人，你可能想：
> - 阅读有关[身份验证概念](https://langchain-ai.github.io/langgraph/concepts/auth/)的更多信息。
> - 查看[API参考](https://langchain-ai.github.io/langgraph/cloud/reference/sdk/python_sdk_ref/)以了解更多身份验证细节。

#### 第 2 部分：使对话私有
在上一部分中，你创建了一个需要令牌认证的聊天机器人，但用户仍能访问彼此的资源（例如线程）。在本教程中，你将扩展该机器人，为每个用户提供私密的对话，通过添加资源级访问控制，确保用户只能看到自己的线程。
<img src="https://i-blog.csdnimg.cn/direct/627aebf8a1e14913a2baa0a551e1766e.png">

### 2.1 前提条件
在开始本教程之前，确保你已经完成第一部分的教程，并且机器人运行无误。

### 2.2 添加资源授权
在上一教程中，Auth 对象允许你注册一个认证函数，LangGraph 平台用它来验证请求中的 Bearer 令牌。现在，你将使用它来注册一个授权处理程序。

授权处理程序是在认证成功后运行的函数。这些处理程序可以为资源添加元数据（例如谁拥有它们）并过滤每个用户可以看到的内容。

更新你的 `src/security/auth.py`，添加一个在每个请求上运行的授权处理程序：

`src/security/auth.py`


In [ ]:
from langgraph_sdk import Auth

# 保留上一教程中的测试用户
VALID_TOKENS = {
    "user1-token": {"id": "user1", "name": "Alice"},
    "user2-token": {"id": "user2", "name": "Bob"},
}

auth = Auth()

@auth.authenticate
async def get_current_user(authorization: str | None) -> Auth.types.MinimalUserDict:
    """上一教程中的认证处理程序。"""
    assert authorization
    scheme, token = authorization.split()
    assert scheme.lower() == "bearer"

    if token not in VALID_TOKENS:
        raise Auth.exceptions.HTTPException(status_code=401, detail="无效令牌")

    user_data = VALID_TOKENS[token]
    return {
        "identity": user_data["id"],
    }

@auth.on
async def add_owner(
    ctx: Auth.types.AuthContext,  # 包含当前用户的信息
    value: dict,  # 被创建或访问的资源
):
    """通过资源元数据使资源对创建者私有。"""
    # 示例：
    # ctx: AuthContext(
    #     permissions=[],
    #     user=ProxyUser(
    #         identity='user1',
    #         is_authenticated=True,
    #         display_name='user1'
    #     ),
    #     resource='threads',
    #     action='create_run'
    # )
    # value: 
    # {
    #     'thread_id': UUID('1e1b2733-303f-4dcd-9620-02d370287d72'),
    #     'assistant_id': UUID('fe096781-5601-53d2-b2f6-0d3403f7e9ca'),
    #     'run_id': UUID('1efbe268-1627-66d4-aa8d-b956b0f02a41'),
    #     'status': 'pending',
    #     'metadata': {},
    #     'prevent_insert_if_inflight': True,
    #     'multitask_strategy': 'reject',
    #     'if_not_exists': 'reject',
    #     'after_seconds': 0,
    #     'kwargs': {
    #         'input': {'messages': [{'role': 'user', 'content': 'Hello!'}]},
    #         'command': None,
    #         'config': {
    #             'configurable': {
    #                 'langgraph_auth_user': ... 你的用户对象 ...
    #                 'langgraph_auth_user_id': 'user1'
    #             }
    #         },
    #         'stream_mode': ['values'],
    #         'interrupt_before': None,
    #         'interrupt_after': None,
    #         'webhook': None,
    #         'feedback_keys': None,
    #         'temporary': False,
    #         'subgraphs': False
    #     }
    # }

    # 执行两件事：
    # 1. 将用户 ID 添加到资源的元数据中。每个 LangGraph 资源都有一个随资源持久化的 `metadata` 字典。
    # 此元数据在读取和更新操作中可用于过滤
    # 2. 返回一个过滤器，仅允许用户看到自己的资源
    filters = {"owner": ctx.user.identity}
    metadata = value.setdefault("metadata", {})
    metadata.update(filters)

    # 只允许用户看到自己的资源
    return filters


处理程序接收两个参数：

1. `ctx（AuthContext）`：包含当前用户的 `user` 信息、用户的 `permissions`、访问的 `resource（“threads”、“crons”、“assistants”）`和执行的 `action（“create”、“read”、“update”、“delete”、“search”、“create_run”）`。
2. value（dict）：正在创建或访问的数据。此字典的内容取决于访问的资源和操作。有关如何实现更精细的访问控制，请参见下面的添加范围授权处理程序。
注意，这个简单的处理程序执行了两件事：

1. 将用户 ID 添加到资源的元数据中。
2. 返回一个元数据过滤器，确保用户只能看到他们拥有的资源。

### 2.3 测试私有对话
测试你的授权。如果设置正确，你将看到所有 ✅ 消息。确保你的开发服务器正在运行（运行 langgraph dev）：



In [ ]:
from langgraph_sdk import get_client

# 为两个用户创建客户端
alice = get_client(
    url="http://localhost:2024",
    headers={"Authorization": "Bearer user1-token"}
)

bob = get_client(
    url="http://localhost:2024",
    headers={"Authorization": "Bearer user2-token"}
)

# Alice 创建一个助手
alice_assistant = await alice.assistants.create()
print(f"✅ Alice 创建了助手：{alice_assistant['assistant_id']}")

# Alice 创建一个线程并聊天
alice_thread = await alice.threads.create()
print(f"✅ Alice 创建了线程：{alice_thread['thread_id']}")

await alice.runs.create(
    thread_id=alice_thread["thread_id"],
    assistant_id="agent",
    input={"messages": [{"role": "user", "content": "嗨，这是 Alice 的私有聊天"}]}
)

# Bob 尝试访问 Alice 的线程
try:
    await bob.threads.get(alice_thread["thread_id"])
    print("❌ Bob 不应该看到 Alice 的线程！")
except Exception as e:
    print("✅ Bob 被正确拒绝访问：", e)

# Bob 创建自己的线程
bob_thread = await bob.threads.create()
await bob.runs.create(
    thread_id=bob_thread["thread_id"],
    assistant_id="agent",
    input={"messages": [{"role": "user", "content": "嗨，这是 Bob 的私有聊天"}]}
)
print(f"✅ Bob 创建了自己的线程：{bob_thread['thread_id']}")

# 列出线程 - 每个用户只看到自己的线程
alice_threads = await alice.threads.search()
bob_threads = await bob.threads.search()
print(f"✅ Alice 看到 {len(alice_threads)} 个线程")
print(f"✅ Bob 看到 {len(bob_threads)} 个线程")


输出：

✅ Alice 创建了助手：fc50fb08-78da-45a9-93cc-1d3928a3fc37                                
✅ Alice 创建了线程：533179b7-05bc-4d48-b47a-a83cbdb5781d                                   
✅ Bob 被正确拒绝访问：Client error '404 Not Found' for url 'http://localhost:2024/threads/      533179b7-05bc-4d48-b47a-a83cbdb5781d'           
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404          
✅ Bob 创建了自己的线程：437c36ed-dd45-4a1e-b484-28ba6eca8819            
✅ Alice 看到 1 个线程             
✅ Bob 看到 1 个线程             


### 2.4 添加范围授权处理程序
广义的 `@auth.on` 处理程序匹配所有授权事件。这很简洁，但意味着 `value` 字典的内容范围不明确，且对每个资源应用相同的用户级访问控制。如果你想要更细粒度的控制，你可以针对特定资源操作进行控制。

更新 `src/security/auth.py`，添加针对特定资源类型的处理程序：


In [ ]:

# 保留之前的处理程序...

from langgraph_sdk import Auth

@auth.on.threads.create
async def on_thread_create(
    ctx: Auth.types.AuthContext,
    value: Auth.types.on.threads.create.value,
):
    """在创建线程时添加所有者。

    此处理程序在创建新线程时运行，执行两件事：
    1. 在创建的线程上设置元数据以跟踪所有权
    2. 返回一个过滤器，确保只有创建者可以访问
    """
    # 示例 value：
    #  {'thread_id': UUID('99b045bc-b90b-41a8-b882-dabc541cf740'), 'metadata': {}, 'if_exists': 'raise'}

    # 在创建的线程上添加所有者元数据
    # 此元数据随线程存储并持久化
    metadata = value.setdefault("metadata", {})
    metadata["owner"] = ctx.user.identity

    # 返回过滤器，仅限创建者访问
    return {"owner": ctx.user.identity}

@auth.on.threads.read
async def on_thread_read(
    ctx: Auth.types.AuthContext,
    value: Auth.types.on.threads.read.value,
):
    """仅允许用户读取自己的线程。

    此处理程序在读取操作时运行。由于线程已存在，无需设置元数据，
    只需返回一个过滤器，确保用户只能看到自己的线程。
    """
    return {"owner": ctx.user.identity}

@auth.on.assistants
async def on_assistants(
    ctx: Auth.types.AuthContext,
    value: Auth.types.on.assistants.value,
):
    # 为了说明目的，我们将拒绝所有涉及助手资源的请求
    # 示例 value：
    # {
    #     'assistant_id': UUID('63ba56c3-b074-4212-96e2-cc333bbc4eb4'),
    #     'graph_id': 'agent',
    #     'config': {},
    #     'metadata': {},
    #     'name': 'Untitled'
    # }
    raise Auth.exceptions.HTTPException(
        status_code=403,
        detail="用户缺乏所需权限。",
    )

# 假设你在存储中按 (user_id, resource_type, resource_id) 组织信息
@auth.on.store()
async def authorize_store(ctx: Auth.types.AuthContext, value: dict):
    # 每个存储项的“namespace”字段是一个可以视为项目录的元组
    namespace: tuple = value["namespace"]
    assert namespace[0] == ctx.user.identity, "未授权"


注意，与其使用一个全局处理程序，你现在为以下操作定义了特定处理程序：
1. 创建线程
2. 读取线程
3. 访问助手

前三个处理程序匹配每个资源上的特定操作（参见资源操作），而最后一个（@auth.on.assistants）匹配助手资源上的任何操作。对于每个请求，LangGraph 将运行与被访问资源和操作匹配的最特定处理程序。这意味着上述四个处理程序将优先于广义的“@auth.on”处理程序运行。

将以下测试代码添加到你的测试文件中：

In [ ]:
# ... 与之前相同
# 尝试创建助手。这应该失败
try:
    await alice.assistants.create("agent")
    print("❌ Alice 不应该能够创建助手！")
except Exception as e:
    print("✅ Alice 被正确拒绝访问：", e)

# 尝试搜索助手。这也应该失败
try:
    await alice.assistants.search()
    print("❌ Alice 不应该能够搜索助手！")
except Exception as e:
    print("✅ Alice 被正确拒绝搜索助手访问：", e)

# Alice 仍然可以创建线程
alice_thread = await alice.threads.create()
print(f"✅ Alice 创建了线程：{alice_thread['thread_id']}")


输出：

```shell

✅ Alice 创建了线程：dcea5cd8-eb70-4a01-a4b6-643b14e8f754
✅ Bob 被正确拒绝访问：Client error '404 Not Found' for url 'http://localhost:2024/threads/dcea5cd8-eb70-4a01-a4b6-643b14e8f754'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
✅ Bob 创建了自己的线程：400f8d41-e946-429f-8f93-4fe395bc3eed
✅ Alice 看到 1 个线程
✅ Bob 看到 1 个线程
✅ Alice 被正确拒绝访问：
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/500
✅ Alice 被正确拒绝搜索助手访问：

```

总结：恭喜！你已经构建了一个为每个用户提供私有对话的聊天机器人。虽然此系统使用简单的基于令牌的认证，但这些授权模式将适用于实现任何真实认证系统。在下一部分，你将用 OAuth2 的真实用户账户替换测试用户。

> 现在您可以控制对资源的访问，您可能希望：       
> 阅读更多关于[授权模式](https://langchain-ai.github.io/langgraph/concepts/auth/#authorization)的信息。                 
> 有关本教程中使用的接口和方法的详细信息，请参阅[API参考](https://langchain-ai.github.io/langgraph/cloud/reference/sdk/python_sdk_ref/#langgraph_sdk.auth.Auth)。

## 第 3 部分：连接认证提供者
在前两部分中，你实现了基于硬编码令牌的认证和资源授权，为用户提供了私有对话。然而，硬编码令牌不安全。现在，你将通过使用 OAuth2 和 Supabase 作为身份提供者，替换这些令牌为真实用户账户。

你将保留相同的 Auth 对象和资源级访问控制，但将认证升级为使用 Supabase 进行验证。虽然本教程使用 Supabase，但这些概念适用于任何 OAuth2 提供者。你将学习如何：

- 用真实的 JWT 令牌替换测试令牌
- 集成 OAuth2 提供者以实现安全的用户认证
- 在保持现有授权逻辑的同时处理用户会话和元数据


标准的 OAuth2 流程如下：

![](https://cdn.mathpix.com/snip/images/Dbp_8Zw-s35jzNTUpBEFHM2ZayPdX1XYiud5X8ORyeM.original.fullsize.png)

### 3.2 前提条件
在开始本教程之前，确保你已：

- 完成第一部分和第二部分的教程，拥有一个运行正常的带有认证和授权的机器人
- 安装了 Supabase CLI 或有权访问 Supabase 仪表板以获取项目 URL 和密钥
- 一个有效的电子邮件地址用于测试用户账户

### 3.3 安装依赖
在你的 custom-auth 目录中安装所需依赖，确保已安装 `langgraph-cli`：

```shell
cd custom-auth
pip install -U "langgraph-cli[inmem]"
```


### 3.4 设置认证提供者
接下来，获取你的认证服务器的 URL 和用于认证的私钥。由于你使用 Supabase，你可以在 Supabase 仪表板中完成以下操作：

在左侧栏中，点击“⚙ 项目设置”，然后点击“API”。

复制你的项目 URL 并添加到 `.env` 文件：

```shell
echo "SUPABASE_URL=your-project-url" >> .env

```

复制你的服务角色密钥并添加到 .env 文件：

```shell
echo "SUPABASE_SERVICE_KEY=your-service-role-key" >> .env

```

复制你的“匿名公共”密钥并记下，稍后在设置客户端代码时将使用它。

`.env` 示例

```plaintext
SUPABASE_URL=your-project-url
SUPABASE_SERVICE_KEY=your-service-role-key
```

### 3.5 实现令牌验证
在之前的教程中，你使用 Auth 对象验证硬编码令牌并添加资源所有权。现在，你将升级认证以验证来自 `Supabase` 的真实 JWT 令牌。主要更改将在 `@auth.authenticate` 装饰的函数中：

- 不检查硬编码令牌列表，而是向 Supabase 发出 HTTP 请求以验证令牌。
- 从验证的令牌中提取真实用户信息（ID、电子邮件）。
- 现有的资源授权逻辑保持不变。

更新 `src/security/auth.py` 以实现此功能：

`src/security/auth.py`




In [ ]:
import os
import httpx
from langgraph_sdk import Auth

auth = Auth()

# 从你上面创建的 `.env` 文件加载
SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_SERVICE_KEY = os.environ["SUPABASE_SERVICE_KEY"]

@auth.authenticate
async def get_current_user(authorization: str | None):
    """验证 JWT 令牌并提取用户信息。"""
    assert authorization
    scheme, token = authorization.split()
    assert scheme.lower() == "bearer"

    try:
        # 使用认证提供者验证令牌
        async with httpx.AsyncClient() as client:
            response = await client.get(
                f"{SUPABASE_URL}/auth/v1/user",
                headers={
                    "Authorization": authorization,
                    "apiKey": SUPABASE_SERVICE_KEY,
                },
            )
            assert response.status_code == 200
            user = response.json()
            return {
                "identity": user["id"],  # 唯一用户标识符
                "email": user["email"],
                "is_authenticated": True,
            }
    except Exception as e:
        raise Auth.exceptions.HTTPException(status_code=401, detail=str(e))

# ... 其余部分与之前相同

# 保留上一教程中的资源授权
@auth.on
async def add_owner(ctx, value):
    """通过资源元数据使资源对创建者私有。"""
    filters = {"owner": ctx.user.identity}
    metadata = value.setdefault("metadata", {})
    metadata.update(filters)
    return filters


最重要的变化是你现在使用真实的认证服务器验证令牌。认证处理程序拥有 Supabase 项目的私钥，可用于验证用户令牌并提取其信息。


### 3.6 测试认证流程
让我们测试新的认证流程。你可以在文件或笔记本中运行以下代码。你需要提供：

- 一个有效的电子邮件地址
- Supabase 项目 URL（从上面获取）
- Supabase 匿名公共密钥（也从上面获取）

In [ ]:
import os
import httpx
from getpass import getpass
from langgraph_sdk import get_client

# 从命令行获取电子邮件
email = getpass("请输入你的电子邮件：")
base_email = email.split("@")
password = "secure-password"  # 请更改此密码
email1 = f"{base_email[0]}+1@{base_email[1]}"
email2 = f"{base_email[0]}+2@{base_email[1]}"

SUPABASE_URL = os.environ.get("SUPABASE_URL")
if not SUPABASE_URL:
    SUPABASE_URL = getpass("请输入你的 Supabase 项目 URL：")

# 这是你的公开匿名密钥（可在客户端安全使用）
# 不要将其与秘密服务角色密钥混淆
SUPABASE_ANON_KEY = os.environ.get("SUPABASE_ANON_KEY")
if not SUPABASE_ANON_KEY:
    SUPABASE_ANON_KEY = getpass("请输入你的 Supabase 公开匿名密钥：")

async def sign_up(email: str, password: str):
    """创建新的用户账户。"""
    async with httpx.AsyncClient() as client:
        response = await client.post(
            f"{SUPABASE_URL}/auth/v1/signup",
            json={"email": email, "password": password},
            headers={"apiKey": SUPABASE_ANON_KEY},
        )
        assert response.status_code == 200
        return response.json()

# 创建两个测试用户
print(f"创建测试用户：{email1} 和 {email2}")
await sign_up(email1, password)
await sign_up(email2, password)


继续之前：检查你的电子邮件并点击两个确认链接。Supabase 在你确认用户电子邮件之前会拒绝 `/login` 请求。

现在测试用户只能看到自己的数据。确保服务器正在运行（运行 langgraph dev）再继续。以下代码片段需要你在设置认证提供者时从 Supabase 仪表板复制的“匿名公共”密钥。


In [ ]:
async def login(email: str, password: str):
    """为现有用户获取访问令牌。"""
    async with httpx.AsyncClient() as client:
        response = await client.post(
            f"{SUPABASE_URL}/auth/v1/token?grant_type=password",
            json={
                "email": email,
                "password": password
            },
            headers={
                "apikey": SUPABASE_ANON_KEY,
                "Content-Type": "application/json"
            },
        )
        assert response.status_code == 200
        return response.json()["access_token"]

# 以用户 1 身份登录
user1_token = await login(email1, password)
user1_client = get_client(
    url="http://localhost:2024", headers={"Authorization": f"Bearer {user1_token}"}
)

# 以用户 1 身份创建线程
thread = await user1_client.threads.create()
print(f"✅ 用户 1 创建了线程：{thread['thread_id']}")

# 尝试不带令牌访问
unauthenticated_client = get_client(url="http://localhost:2024")
try:
    await unauthenticated_client.threads.create()
    print("❌ 未认证访问应该失败！")
except Exception as e:
    print("✅ 未认证访问被阻止：", e)

# 尝试以用户 2 身份访问用户 1 的线程
user2_token = await login(email2, password)
user2_client = get_client(
    url="http://localhost:2024", headers={"Authorization": f"Bearer {user2_token}"}
)

try:
    await user2_client.threads.get(thread["thread_id"])
    print("❌ 用户 2 不应该看到用户 1 的线程！")
except Exception as e:
    print("✅ 用户 2 被阻止访问用户 1 的线程：", e)


输出应如下所示：
```shell
✅ 用户 1 创建了线程：d6af3754-95df-4176-aa10-dbd8dca40f1a
✅ 未认证访问被阻止：Client error '403 Forbidden' for url 'http://localhost:2024/threads'
✅ 用户 2 被阻止访问用户 1 的线程：Client error '404 Not Found' for url 'http://localhost:2024/threads/d6af3754-95df-4176-aa10-dbd8dca40f1a'

```


你的认证和授权协同工作：

1. 用户必须登录才能访问机器人。
2. 每个用户只能看到自己的线程。

所有用户由 Supabase 认证提供者管理，因此你无需实现额外的用户管理逻辑。